In [ ]:
!git clone https://github.com/VikramShenoy97/Human-Segmentation-Dataset.git

fatal: destination path 'Human-Segmentation-Dataset' already exists and is not an empty directory.


In [ ]:
import os
import time
import torch
from PIL import Image
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader,Dataset
from torch.optim import Adam, AdamW,SGD

In [ ]:
class SegmentationDataset(Dataset):
  def __init__(self,image_dir,mask_dir):
    self.image_dir=image_dir
    self.mask_dir=mask_dir
    self.transform=transforms.Compose([
        transforms.Resize((512,512)),
        transforms.ToTensor()
    ])
    valid_extension={".jpg",".jpeg",".png"}
    self.images=[img_file for img_file in os.listdir(image_dir) if os.path.splitext(img_file)[1].lower() in valid_extension]

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image_path=os.path.join(self.image_dir,self.images[idx])
    name,ext=os.path.splitext(self.images[idx])
    mask_path=os.path.join(self.mask_dir,f'{name}.png')
    img=Image.open(image_path).convert('RGB')
    mask=Image.open(mask_path).convert('L')
    img=self.transform(img)
    mask=self.transform(mask)
    mask=(mask>0.5).float()
    return img,mask



In [ ]:
def getdataloader(image_dir,mask_dir,batch_size=2,shuffle=True):
  dataset=SegmentationDataset(image_dir,mask_dir)
  return DataLoader(dataset,batch_size=batch_size,shuffle=shuffle)


In [ ]:
class DoubleConv(nn.Module):
  def __init__(self,in_channels,out_channels):
    super().__init__()
    self.conv_op=nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(inplace=True)
    )

  def forward(self,x):
    return self.conv_op(x)


In [ ]:
class DownSample(nn.Module):
  def __init__(self,in_channels,out_channels):
    super().__init__()
    self.conv=DoubleConv(in_channels,out_channels)
    self.pool=nn.MaxPool2d(kernel_size=2,stride=2)

  def forward(self,x):
    down=self.conv(x)
    p=self.pool(down)
    return down,p

In [ ]:
class UpSample(nn.Module):
  def __init__(self,in_channels,out_channels):
    super().__init__()
    self.up=nn.ConvTranspose2d(in_channels,in_channels//2,kernel_size=2,stride=2)
    self.conv=DoubleConv(in_channels,out_channels)

  def forward(self,x1,x2):
    x1=self.up(x1)
    x=torch.cat([x1,x2],1)

    return self.conv(x)

In [ ]:
class Unet(nn.Module):
  def __init__(self,in_channels,num_classes):
    super(Unet,self).__init__()
    self.down_conv1=DownSample(in_channels,64)
    self.down_conv2=DownSample(64,128)
    self.down_conv3=DownSample(128,256)
    self.down_conv4=DownSample(256,512)

    self.bottleneck=DoubleConv(512,1024)

    self.up_conv1=UpSample(1024,512)
    self.up_conv2=UpSample(512,256)
    self.up_conv3=UpSample(256,128)
    self.up_conv4=UpSample(128,64)

    self.out=nn.Conv2d(in_channels=64,out_channels=num_classes,kernel_size=1)

  def forward(self,x):
    down1,p1=self.down_conv1(x)
    down2,p2=self.down_conv2(p1)
    down3,p3=self.down_conv3(p2)
    down4,p4=self.down_conv4(p3)
    b=self.bottleneck(p4)
    up1=self.up_conv1(b,down4)
    up2=self.up_conv2(up1,down3)
    up3=self.up_conv3(up2,down2)
    up4=self.up_conv4(up3,down1)

    return self.out(up4)

In [ ]:
class DiceLoss(nn.Module):
  def __init__(self,smooth=1e-6):
    super(DiceLoss,self).__init__()
    self.smooth=smooth
  def forward(self,inputs,targets):
    inputs=inputs.view(-1)
    targets=targets.view(-1)
    intersection=(inputs*targets).sum()
    dice_score=(2.*intersection+self.smooth)/(inputs.sum()+targets.sum()+self.smooth)
    return 1-dice_score

In [ ]:
class BCEWithDiceLoss(nn.Module):
  def __init__(self,smooth=1e-6):
    super(BCEWithDiceLoss,self).__init__()
    self.bce=nn.BCEWithLogitsLoss()
    self.dice=DiceLoss(smooth)

  def forward(self,inputs,targets):
    bce_loss=self.bce(inputs,targets)
    dice_loss=self.dice(inputs,targets)
    return bce_loss+dice_loss

In [ ]:
#Training loop
def train(model,dataloader,epochs=2,lr=0.001,save_path="unet_model",load_path=None):
  torch.cuda.empty_cache()
  device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
  if load_path and os.path.exists(load_path):
    print(f"Loading model from {load_path}")
    model.load_state_dict(torch.load(load_path,map_location=device))
  else:
    print("No checkpoint found. Training from scratch.")

  print(device)
  model.to(device)

  optimizer=SGD(model.parameters(),lr=lr)
  criterion=BCEWithDiceLoss()

  for epoch in range(epochs):
    model.train()
    epoch_loss=0
    for images,masks in dataloader:
      images,masks=images.to(device),masks.to(device)
      optimizer.zero_grad()
      output=model(images)
      loss=criterion(output,masks)
      loss.backward()
      optimizer.step()
      epoch_loss+=loss.item()
    avg_loss=epoch_loss/len(dataloader)
    print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, LR={lr}')

    if epoch>0 and epoch%10==0:
      torch.save(model.state_dict(),f'{save_path}.pth')

  torch.save(model.state_dict(),f'{save_path}.pth')
  print(f"Model saved to {save_path}.")

In [ ]:
dataloader=getdataloader('/content/Human-Segmentation-Dataset/Training_Images','/content/Human-Segmentation-Dataset/Ground_Truth',batch_size=4)

In [ ]:
model=Unet(in_channels=3,num_classes=1)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
train(model,dataloader,epochs=10,lr=0.001)

No checkpoint found. Training from scratch.
cuda
Epoch 1/10, Loss: 1.5195, LR=0.001
Epoch 2/10, Loss: 1.5053, LR=0.001
Epoch 3/10, Loss: 1.4972, LR=0.001
Epoch 4/10, Loss: 1.4891, LR=0.001
Epoch 5/10, Loss: 1.4871, LR=0.001
Epoch 6/10, Loss: 1.4828, LR=0.001
Epoch 7/10, Loss: 1.4828, LR=0.001
Epoch 8/10, Loss: 1.4832, LR=0.001
Epoch 9/10, Loss: 1.4797, LR=0.001
Epoch 10/10, Loss: 1.4779, LR=0.001
Model saved to unet_model.


In [36]:
import numpy as np

def predict(model_path,input_image_path):
  device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f'Using device: {device}')
  model=Unet(in_channels=3,num_classes=1)
  model.load_state_dict(torch.load(model_path,map_location=device))
  model.to(device)
  model.eval()

  total_start_time=time.time()
  #image preprocessing
  preprocess_start_time=time.time()
  image=Image.open(input_image_path).convert('RGB')
  transform=transforms.Compose([
      transforms.Resize((512,512)),
      transforms.ToTensor()
  ])
  image_tensor=transform(image).unsqueeze(0).to(device)
  preprocess_end_time=time.time()
  #Model infrence
  infrence_start_time=time.time()
  with torch.no_grad():
    output=model(image_tensor)
    output=torch.sigmoid(output)
  infrence_end_time=time.time()
  #Postprocessing
  postprocess_start_time=time.time()
  mask=output.squeeze(0).squeeze(0).cpu().numpy()
  mask=(mask>0.4).astype(np.uint8)*255
  mask_image=Image.fromarray(mask)

  combined=Image.new("RGB",(512*2,512))
  combined.paste(image.resize((512,512)),(0,0))
  combined.paste(mask_image.convert("RGB"),(512,0))
  combined.save("output.jpg")
  postprocess_end_time=time.time()

  total_end_time=time.time()

  preprocess_time=preprocess_end_time-preprocess_start_time
  infrence_time=infrence_end_time-infrence_start_time
  postprocess_time=postprocess_end_time-postprocess_start_time
  total_time=total_end_time-total_start_time

  print(f'Preprocessing time: {preprocess_time:.4f} seconds')
  print(f'Inference time: {infrence_time:.4f} seconds')
  print(f'Postprocessing time: {postprocess_time:.4f} seconds')
  print(f'Total time: {total_time:.4f} seconds')
  print("Prediction saved as output.jpg")







In [37]:
predict('/content/unet_model.pth','/content/Human-Segmentation-Dataset/Training_Images/10.jpg')

Using device: cuda
Preprocessing time: 0.0088 seconds
Inference time: 0.0026 seconds
Postprocessing time: 0.1231 seconds
Total time: 0.1346 seconds
Prediction saved as output.jpg
